# Trabalho Final — Recomendação de Consultorias Sebrae RN

Lucas Medeiros — Aprendizado Profundo, PPgTI/IMD/UFRN, Prof. Josenalde Oliveira

## O que este notebook faz

A ideia é apoiar o time comercial do Sebrae/RN em duas decisões: para quais empresas vale oferecer
uma nova consultoria, e qual tema oferecer. São duas redes em cascata:

- **Rede 1 (binária):** a empresa tem propensão a contratar alguma consultoria?
- **Rede 2 (multiclasse):** entre quem contrata, qual dos 8 grupos temáticos recomendar?

Para cada uma das redes são treinadas duas arquiteturas: um **modelo base** (MLP com as variáveis
categóricas codificadas por frequência) e um **modelo com embeddings** (a própria rede aprende um
vetor para cada categoria). No final as duas são comparadas no mesmo conjunto de teste, e a cascata
é avaliada de ponta a ponta.

## 1. Setup

In [ ]:
# Instala tudo que o notebook usa. Em ambiente que ja tem os pacotes, o pip so confirma e segue.
import sys

PACOTES = [
    "numpy", "pandas", "openpyxl",      # dados
    "scikit-learn",                      # pre-processamento, metricas, baseline
    "tensorflow",                        # redes neurais
    "optuna",                            # busca de hiperparametros
    "matplotlib", "pillow",              # graficos e o gif de treinamento
    "joblib", "dill",                    # salvar modelo e pre-processador
]

print("Instalando dependencias...")
!{sys.executable} -m pip install -q {" ".join(PACOTES)}
print("Ambiente pronto.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    recall_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve,
)

import joblib

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)
optuna.logging.set_verbosity(optuna.logging.WARNING)   # so o resumo, sem log de cada trial

# semente fixa pra deixar a execucao reproduzivel
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Se a base nao estiver na mesma pasta, baixa do repositorio publico no GitHub.
# Assim o notebook roda em qualquer ambiente sem configuracao manual.
NOME_ARQUIVO_BASE = 'base_modelagem_anonimizada.xlsx'
URL_BASE_GITHUB = (
    'https://raw.githubusercontent.com/lucassmsantoss/aprendizado-profundo/'
    'main/trabalho_final_recomendacao_consultorias/base_modelagem_anonimizada.xlsx'
)

if not os.path.exists(NOME_ARQUIVO_BASE):
    print("Base nao encontrada localmente - baixando do repositorio no GitHub...")
    import urllib.request
    urllib.request.urlretrieve(URL_BASE_GITHUB, NOME_ARQUIVO_BASE)
    print("Download concluido.")

CAMINHO_XLSX = NOME_ARQUIVO_BASE
PASTA_MODELOS = 'modelos'
PASTA_FIGURAS = 'figuras'
os.makedirs(PASTA_MODELOS, exist_ok=True)
os.makedirs(PASTA_FIGURAS, exist_ok=True)

# Se ja existir modelo salvo na pasta, o notebook reaproveita em vez de treinar de novo.
# Deixe False pra forcar o treinamento completo do zero.
REUSAR_MODELOS_SALVOS = False

## 2. Os dados

A base tem 220.168 empresas atendidas pelo Sebrae/RN, enriquecidas com dados cadastrais da Receita
Federal. Cada linha é uma empresa. Antes de modelar, vale olhar o que tem aqui dentro.

In [ ]:
df = pd.read_excel(CAMINHO_XLSX, sheet_name='Base modelagem')

# papel de cada coluna no problema, pra deixar claro o que e feature e o que e alvo
PAPEL_COLUNAS = {
    'ID_EMPRESA': 'identificador (nao entra no modelo)',
    'Qtd de atendimentos PJ': 'feature numerica',
    'Qtd de projetos atendidos': 'feature numerica',
    'Qtd de consultorias contratadas': 'nao usada (vaza a resposta da Rede 1)',
    'Target - consultoria mais recente': 'origem da target da Rede 2',
    'Contratou consultoria': 'target da Rede 1',
    'ds_sebrae_segmento': 'feature categorica',
    'nm_municipio': 'feature categorica',
    'idade_empresa_meses': 'feature numerica',
    'ds_tipo_estabelecimento': 'feature categorica',
    'sg_porte': 'feature categorica',
}

tabela_variaveis = pd.DataFrame({
    'papel no modelo': [PAPEL_COLUNAS[c] for c in df.columns],
    'tipo': [str(df[c].dtype) for c in df.columns],
    'valores unicos': [df[c].nunique() for c in df.columns],
    '% nulos': [round(df[c].isna().mean() * 100, 2) for c in df.columns],
})
tabela_variaveis.index = df.columns
tabela_variaveis.index.name = 'variavel'

print(f"Base bruta: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n")
tabela_variaveis

Os 5,7% de nulos aparecem sempre nas mesmas 5 colunas e nas mesmas linhas: são empresas que não
foram encontradas no enriquecimento cadastral. Como são justamente as variáveis de perfil da
empresa, essas linhas não têm como ser usadas e saem da base.

In [ ]:
COLS_ENRIQUECIMENTO = ['ds_sebrae_segmento', 'nm_municipio', 'idade_empresa_meses',
                        'ds_tipo_estabelecimento', 'sg_porte']

# confere que as 5 colunas ficam nulas juntas, nas mesmas linhas
nulos_por_linha = df[COLS_ENRIQUECIMENTO].isna().sum(axis=1)
print("Colunas de enriquecimento nulas por linha:")
print(nulos_por_linha.value_counts().rename_axis('qtd de colunas nulas').to_frame('linhas'))

df_limpo = df.dropna(subset=COLS_ENRIQUECIMENTO).copy()
print(f"\nLinhas antes da limpeza: {df.shape[0]:,} | apos a limpeza: {df_limpo.shape[0]:,}")

In [ ]:
FEATURES_NUMERICAS = ['Qtd de atendimentos PJ', 'Qtd de projetos atendidos', 'idade_empresa_meses']
FEATURES_CATEGORICAS = ['ds_sebrae_segmento', 'nm_municipio', 'ds_tipo_estabelecimento', 'sg_porte']

# estatisticas das variaveis numericas depois da limpeza
tabela_numericas = df_limpo[FEATURES_NUMERICAS].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
tabela_numericas = tabela_numericas.round(2)
tabela_numericas.index.name = 'variavel numerica'
tabela_numericas

As três variáveis numéricas são bem assimétricas — a mediana de atendimentos é 4 e o máximo passa
de 12 mil. É por isso que o pré-processamento aplica `log1p` antes de padronizar: sem isso, um
punhado de empresas gigantes domina a escala.

In [ ]:
# distribuicao da target da Rede 1
contagem_alvo = df_limpo['Contratou consultoria'].map({0: 'Nao contratou', 1: 'Contratou'}).value_counts()
tabela_alvo_r1 = pd.DataFrame({
    'empresas': contagem_alvo,
    '% do total': (contagem_alvo / contagem_alvo.sum() * 100).round(2),
})
tabela_alvo_r1.index.name = 'target da Rede 1'
print("Desbalanceamento da Rede 1: so 1 em cada 40 empresas contratou alguma consultoria.\n")
tabela_alvo_r1

In [ ]:
# distribuicao das variaveis categoricas
for coluna in ['sg_porte', 'ds_tipo_estabelecimento']:
    contagem = df_limpo[coluna].value_counts()
    tabela = pd.DataFrame({'empresas': contagem, '% do total': (contagem / contagem.sum() * 100).round(2)})
    tabela.index.name = coluna
    print(tabela.to_string(), '\n')

print(f"nm_municipio: {df_limpo['nm_municipio'].nunique()} municipios distintos (5 maiores)")
print(df_limpo['nm_municipio'].value_counts().head(5).to_string(), '\n')
print(f"ds_sebrae_segmento: {df_limpo['ds_sebrae_segmento'].nunique()} segmentos distintos (5 maiores)")
print(df_limpo['ds_sebrae_segmento'].value_counts().head(5).to_string())

O número de categorias é o que justifica a escolha de codificação mais à frente: `nm_municipio`
sozinho tem 167 valores distintos. Um one-hot criaria 167 colunas só para essa variável.

## 3. Agrupamento temático da target (8 classes)

A target bruta (`Target - consultoria mais recente`) tem 116 categorias, e a distribuição é bem
desigual: poucas categorias concentram a maioria dos casos, e muitas outras têm só 1 ou 2 exemplos
no total. Não dá pra treinar uma rede em cima de 116 classes assim — a maior parte delas nunca
teria exemplo suficiente pra aprender nada.

Então agrupei as 116 categorias em 8 grupos temáticos, seguindo a mesma lógica do Data Product
Canvas do projeto. O `GRUPO_MAP` abaixo é esse de-para completo ("Não contratou consultoria" é um
rótulo à parte, usado só pela Rede 1).

In [ ]:
GRUPO_MAP = {
    "CNPJ não contratou nenhuma consultoria": "Não contratou consultoria",
    "Saúde e Segurança no Trabalho – PGR (NR-1), PCMSO, LTCAT, Laudo de Insalubridade e Laudo de Periculosidade no E-social": "Saúde e Segurança no Trabalho",
    "Saúde e Segurança no Trabalho 02 – PGR (NR-1), PCMSO, LTCAT, Laudo de Insalubridade e Laudo de Periculosidade no E-social": "Saúde e Segurança no Trabalho",
    "Consultoria em Saúde e Segurança no Trabalho – Diagnóstico de NR´s": "Saúde e Segurança no Trabalho",
    "Adequação à NR-17 - Ergonomia": "Saúde e Segurança no Trabalho",
    "Adequação à NR 35 - Trabalho em Altura": "Saúde e Segurança no Trabalho",
    "Adequação à NR 10 – Instalações Elétricas": "Saúde e Segurança no Trabalho",
    "Avaliação Ambiental - Agentes Químicos (Higiene Ocupacional)": "Saúde e Segurança no Trabalho",
    "Avaliação Ambiental - Agentes Físicos (Vibração)": "Saúde e Segurança no Trabalho",
    "Projeto de Combate a Incêndio e Pânico": "Saúde e Segurança no Trabalho",
    "Licenciamento Ambiental": "Meio Ambiente",
    "Plano de Gerenciamento de Resíduos Sólidos": "Meio Ambiente",
    "Sustentabilidade Ambiental, Econômica e Social na Mineração": "Meio Ambiente",
    "Gestão de Efluentes Líquidos": "Meio Ambiente",
    "Plano de Recuperação de Áreas Degradadas (PRAD)": "Meio Ambiente",
    "Outorga de Água subterrânea": "Meio Ambiente",
    "Cadastro Ambiental Rural (CAR)": "Meio Ambiente",
    "Consultoria para Estudo de Impacto de Vizinhança": "Meio Ambiente",
    "Emissões Atmosféricas de Fonte Fixa": "Meio Ambiente",
    "Consultoria para Implantação de uma Unidade de Processamento de Matéria Orgânica - Compostagem": "Meio Ambiente",
    "Planejamento para implantação de ações de Responsabilidade Social e Ambiental": "Meio Ambiente",
    "Energia Solar Fotovoltaica": "Meio Ambiente",
    "Design de Ambientes": "Design, Marca e Ambientes",
    "Branding": "Design, Marca e Ambientes",
    "Comunicação Visual": "Design, Marca e Ambientes",
    "Design e Melhoria de Serviços": "Design, Marca e Ambientes",
    "Design de rótulo(s) e aplicações de elementos gráficos na embalagem": "Design, Marca e Ambientes",
    "Desenvolvimento de Coleções": "Design, Marca e Ambientes",
    "Design de Embalagens": "Design, Marca e Ambientes",
    "Modelagem, Encaixe e Plotagem": "Design, Marca e Ambientes",
    "Modelagem e Graduação para Vestuário": "Design, Marca e Ambientes",
    "Boas Práticas em Ambientes Comerciais - Layout e Aspectos do Visual Merchandising": "Design, Marca e Ambientes",
    "Melhoria de Layout Produtivo": "Design, Marca e Ambientes",
    "Quiosque de Venda": "Design, Marca e Ambientes",
    "Desenvolvimento de Mídias Digitais de Comunicação": "Marketing e Presença Digital",
    "Inserção Digital – Desenvolvimento de Website": "Marketing e Presença Digital",
    "Planejamento Para Presença Digital e Links Patrocinados": "Marketing e Presença Digital",
    "UX - Experiência do Usuário em Ambientes Digitais": "Marketing e Presença Digital",
    "Planejamento Para Busca Orgânica – Seo": "Marketing e Presença Digital",
    "Implantação de Loja Virtual": "Marketing e Presença Digital",
    "Marketing Digital - Consultoria": "Marketing e Presença Digital",
    "Implantação de ferramentas do whatsapp para automação de vendas e gestão comercial": "Marketing e Presença Digital",
    "Planejamento e preparação para comercialização em marketplace": "Marketing e Presença Digital",
    "Consultoria Para Growth Hacking": "Marketing e Presença Digital",
    "Implantação do Código de Barra": "Marketing e Presença Digital",
    "Gestão de Negócios Baseados em Análise e Inteligência em Dados": "Dados, Inovação e Tecnologia",
    "Otimização de Processos com Conectividade (IoT)": "Dados, Inovação e Tecnologia",
    "Implantação de Processos de Gestão da Inovação": "Dados, Inovação e Tecnologia",
    "Elaboração de Projeto de Inovação": "Dados, Inovação e Tecnologia",
    "Adequação à Lei Geral de Proteção de Dados (LGPD)": "Dados, Inovação e Tecnologia",
    "Planejamento Estratégico Tecnológico": "Dados, Inovação e Tecnologia",
    "Depósito de Patente de Invenção ou de Modelo de Utilidade": "Dados, Inovação e Tecnologia",
    "Realização de Modelagem e Simulações de projetos em BIM para o Setor da Construção Civil": "Dados, Inovação e Tecnologia",
    "Controle e Melhoria de Processos": "Gestão de Processos e Qualidade",
    "Framework OKR Para Otimização de Processos": "Gestão de Processos e Qualidade",
    "Lean Manufacturing": "Gestão de Processos e Qualidade",
    "Procedimento Operacional Padrão - Pop": "Gestão de Processos e Qualidade",
    "Organização e Controle de Estoque": "Gestão de Processos e Qualidade",
    "Adequação à norma ABNT NBR ISO 9001:2015 - Sistema de Gestão da Qualidade": "Gestão de Processos e Qualidade",
    "Metrologia - Ensaios": "Gestão de Processos e Qualidade",
    "Metrologia - Calibração": "Gestão de Processos e Qualidade",
    "Implantação de Requisitos de Qualidade, Meio Ambiente, Saúde e Segurança no Trabalho, Eficiência Operacional, Eficiência Energética e Compliance para Fornecedores": "Gestão de Processos e Qualidade",
    "Gestão de Processos Empresariais - Consultoria": "Gestão de Processos e Qualidade",
    "Planejamento e controle de produção": "Gestão de Processos e Qualidade",
    "Implantação de Sistemas de Gestão Integrado": "Gestão de Processos e Qualidade",
    "Dimensionamento da capacidade produtiva": "Gestão de Processos e Qualidade",
    "Produtividade – 5S": "Gestão de Processos e Qualidade",
    "Redução de Desperdício na Cozinha": "Gestão de Processos e Qualidade",
    "Redução de Desperdício nos Pequenos Negócios": "Gestão de Processos e Qualidade",
    "Certificação Conforme Programa da Associação Brasileira do Varejo Têxtil - ABVTEX": "Gestão de Processos e Qualidade",
    "Adequação ao Programa Brasileiro da Qualidade e Produtividade do Habitat (PBQP-H)": "Gestão de Processos e Qualidade",
    "Sistema APPCC – Análise de Perigos e Pontos Críticos de Controle": "Gestão de Processos e Qualidade",
    "Adequação Conforme Protocolo GlobalGAP": "Gestão de Processos e Qualidade",
    "Implantação dos requisitos da norma OSHAS 18001 / ISO 45001": "Gestão de Processos e Qualidade",
    "Certificado de Registro Cadastral - CRC - (Setor Petróleo)": "Gestão de Processos e Qualidade",
    "Implantação da Integração de Sistemas Produtivos": "Gestão de Processos e Qualidade",
    "Processos de Governança em Meios de Hospedagem": "Gestão de Processos e Qualidade",
    "Gestão Econômico/Financeira - Consultoria": "Gestão Financeira e Estratégica",
    "Planejamento Estratégico - Consultoria": "Gestão Financeira e Estratégica",
    "Plano de Negócio - Consultoria": "Gestão Financeira e Estratégica",
    "Projetos de viabilidade - Consultoria": "Gestão Financeira e Estratégica",
    "Provimento - Consultoria": "Gestão Financeira e Estratégica",
    "Tributação para Pequenos Negócios - Consultoria": "Gestão Financeira e Estratégica",
    "Contabilidade Financeira e Fiscal - Consultoria": "Gestão Financeira e Estratégica",
    "toria - Monitoramento Financeiro": "Gestão Financeira e Estratégica",
    "Comércio Exterior - Consultoria": "Gestão Financeira e Estratégica",
    "Formatação da Franquia": "Gestão Financeira e Estratégica",
    "Direito Civil - Consultoria": "Gestão Financeira e Estratégica",
    "Cooperação - Consultoria": "Gestão Financeira e Estratégica",
    "Carreira, Remuneração, Acompanhamento e Avaliação de Desempenho e de Resultados - Consultorias": "Gestão Financeira e Estratégica",
    "Desenvolvimento e Treinamento de Pessoas - Consultoria": "Gestão Financeira e Estratégica",
    "Cultura e Clima Organizacional - Consultoria": "Gestão Financeira e Estratégica",
    "Liderança - Consultoria": "Gestão Financeira e Estratégica",
    "Planejamento Estratégico de Pessoal - Consultoria": "Gestão Financeira e Estratégica",
    "toria - Monitoramento em Gestão de Pessoas": "Gestão Financeira e Estratégica",
    "Vendas - Consultoria": "Marketing e Presença Digital",
    "Marketing Estratégico - Consultoria": "Marketing e Presença Digital",
    "Adequação de agroindústrias aos Serviços de Inspeção de Produtos de Origem Animal e/ou Vegetal": "Agronegócio e Alimentos",
    "Elaboração de Cardápio E/ou Fichas Técnicas Para Segmentos de Alimentação": "Agronegócio e Alimentos",
    "Melhoria de Processo Produtivo para o Cultivo de Camarão e/ou Peixe": "Agronegócio e Alimentos",
    "Boas Práticas de Higiene e Segurança Dos Alimentos Para o Setor de Alimentos e Bebidas": "Agronegócio e Alimentos",
    "Adequação da Área de Produção à Legislação Sanitária": "Agronegócio e Alimentos",
    "Rotulagem de Alimentos e Informação Nutricional": "Agronegócio e Alimentos",
    "Boas práticas agrícolas": "Agronegócio e Alimentos",
    "Georreferenciamento do Empreendimento Rural": "Agronegócio e Alimentos",
    "Melhoria de Processo de Produção Para o Segmento de Alimentação": "Agronegócio e Alimentos",
    "Melhoria Genética - Caprinos e Ovinos": "Agronegócio e Alimentos",
    "Boas Práticas na Apicultura e na Meliponicultura": "Agronegócio e Alimentos",
    "Certificação de Produtos Orgânicos": "Agronegócio e Alimentos",
    "Adequação à regulamentação da produção orgânica": "Agronegócio e Alimentos",
    "Implantação de Projeto de Produção Aquícola": "Agronegócio e Alimentos",
    "Inseminação Artificial por Tempo Fixo – IATF – Rebanho": "Agronegócio e Alimentos",
    "Boas Práticas na Avicultura": "Agronegócio e Alimentos",
    "Boas Práticas na Pecuária de Leite e/ou Corte": "Agronegócio e Alimentos",
    "Desenvolvimento de Novos Produtos Alimentícios": "Agronegócio e Alimentos",
    "Boas Práticas no Segmento de Beleza": "Design, Marca e Ambientes"
}

# aplica o de-para e confere se sobrou alguma categoria sem grupo
df_limpo['target_grupo'] = df_limpo['Target - consultoria mais recente'].map(GRUPO_MAP)
nao_mapeados = df_limpo.loc[df_limpo['target_grupo'].isna(), 'Target - consultoria mais recente'].unique()
print(f"Categorias sem mapeamento (deveria ser vazio): {list(nao_mapeados)}")

contagem_grupos = df_limpo['target_grupo'].value_counts()
tabela_grupos = pd.DataFrame({
    'empresas': contagem_grupos,
    '% do total': (contagem_grupos / contagem_grupos.sum() * 100).round(2),
})
tabela_grupos.index.name = 'grupo tematico'
tabela_grupos

Mesmo depois do agrupamento o desbalanceamento continua grande. Entre quem contratou, a maior
classe ("Saúde e Segurança no Trabalho") tem 2.309 empresas e a menor ("Dados, Inovação e
Tecnologia") tem 97 — quase 24 vezes menos. É isso que motiva o peso por classe usado no
treinamento das duas redes.

## 4. Checagem de vazamento de dado

Antes de montar a Rede 1, testei se `Qtd de consultorias contratadas` entrega a resposta.

In [ ]:
# se os zeros ficarem so nas diagonais opostas, a coluna entrega a resposta
print(pd.crosstab(df_limpo['Contratou consultoria'], df_limpo['Qtd de consultorias contratadas'] > 0))

É vazamento perfeito: `Qtd de consultorias contratadas > 0` acontece exatamente quando
`Contratou consultoria == 1`. Por isso essa coluna não é usada como feature em nenhuma das duas
redes.

Vale explicar por que ela sai também da Rede 2, já que ali a população é só de quem contratou e a
coluna deixaria de ser vazamento. O motivo é o uso final: no produto, a Rede 2 vai receber empresas
encaminhadas pela Rede 1, incluindo empresas que nunca contrataram nada (valor 0). Como no treino
da Rede 2 essa coluna nunca vale 0, o modelo estaria extrapolando justamente no caso mais comum de
uso. Deixando ela de fora, as duas redes passam a usar exatamente o mesmo conjunto de variáveis.

## 5. Funções compartilhadas

Aqui ficam as peças usadas pelas duas redes: o pré-processamento, as duas arquiteturas que vão ser
comparadas, as métricas e o esqueleto da busca de hiperparâmetros.

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """troca cada categoria pela frequencia dela no treino (sem one-hot, sem embeddings)"""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        # guarda a frequencia de cada categoria vista no treino
        self.freq_maps_ = {
            col: X[col].astype('object').value_counts(normalize=True).to_dict()
            for col in X.columns
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # categoria que nao apareceu no treino vira 0
        dados = {
            col: X[col].astype('object').map(self.freq_maps_[col]).astype('float64').fillna(0.0)
            for col in self.feature_names_in_
        }
        return pd.DataFrame(dados, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_in_, dtype=object)


def montar_pre_processador(features_numericas, features_categoricas):
    # numericas: log1p pra achatar a cauda das contagens, depois padroniza
    pipeline_numericas = Pipeline([
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('escala', StandardScaler()),
    ])
    # categoricas: viram frequencia e tambem sao padronizadas
    pipeline_categoricas = Pipeline([
        ('frequencia', FrequencyEncoder()),
        ('escala', StandardScaler()),
    ])
    return ColumnTransformer([
        ('num', pipeline_numericas, features_numericas),
        ('cat', pipeline_categoricas, features_categoricas),
    ]).set_output(transform='pandas')

### As duas arquiteturas comparadas

**Modelo base:** um MLP em funil sobre as variáveis já codificadas por frequência. É uma rede
pequena, rápida de treinar, e serve de referência.

**Modelo com embeddings:** em vez de resumir cada categoria a um número (a frequência dela), a
própria rede aprende um vetor para cada município, segmento, porte e tipo de estabelecimento. Esses
vetores são concatenados com as variáveis numéricas e seguem para as camadas densas. É a técnica
padrão de deep learning para dados tabulares e permite que a rede descubra sozinha que dois
municípios se parecem, coisa que a codificação por frequência não consegue representar.

In [ ]:
def construir_modelo_base(dim_entrada, n_classes, n_camadas, unidades_iniciais,
                          usar_batchnorm, taxa_dropout, learning_rate, otimizador_nome):
    """MLP simples sobre as features ja codificadas por frequencia"""
    modelo = keras.Sequential()
    modelo.add(keras.Input(shape=(dim_entrada,)))

    # camadas em funil: cada uma com metade dos neuronios da anterior
    unidades = unidades_iniciais
    for _ in range(n_camadas):
        modelo.add(layers.Dense(unidades, kernel_initializer='he_normal'))
        if usar_batchnorm:
            modelo.add(layers.BatchNormalization())
        modelo.add(layers.Activation('relu'))
        if taxa_dropout > 0:
            modelo.add(layers.Dropout(taxa_dropout))
        unidades = max(unidades // 2, n_classes)

    # saida com uma probabilidade por classe
    modelo.add(layers.Dense(n_classes, activation='softmax'))
    modelo.compile(optimizer=_montar_otimizador(otimizador_nome, learning_rate),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelo


def _montar_otimizador(nome, learning_rate):
    if nome == 'adamw':
        return keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    return keras.optimizers.Adam(learning_rate=learning_rate)


class CodificadorOrdinal:
    """mapeia cada categoria pra um indice inteiro; o indice 0 fica reservado pra
    categoria que nao apareceu no treino (o modelo com embeddings precisa de indices)"""

    def fit(self, df, colunas):
        self.colunas = list(colunas)
        self.mapas_ = {}
        for col in self.colunas:
            categorias = sorted(df[col].astype(str).unique())
            self.mapas_[col] = {cat: i + 1 for i, cat in enumerate(categorias)}
        # +1 no tamanho por causa do indice 0 (categoria desconhecida)
        self.tamanhos_ = {col: len(mapa) + 1 for col, mapa in self.mapas_.items()}
        return self

    def transform(self, df):
        return {
            col: df[col].astype(str).map(self.mapas_[col]).fillna(0).astype('int32').values.reshape(-1, 1)
            for col in self.colunas
        }


def preparar_dados_embeddings(treino, val, teste, features_numericas, features_categoricas):
    """monta as entradas do modelo com embeddings: um vetor de indices por variavel
    categorica, mais um bloco com as numericas ja padronizadas"""
    pipeline_num = Pipeline([
        ('log1p', FunctionTransformer(np.log1p)),
        ('escala', StandardScaler()),
    ])
    # de novo: fit so no treino
    num_treino = pipeline_num.fit_transform(treino[features_numericas]).astype('float32')
    num_val = pipeline_num.transform(val[features_numericas]).astype('float32')
    num_teste = pipeline_num.transform(teste[features_numericas]).astype('float32')

    codificador = CodificadorOrdinal().fit(treino, features_categoricas)

    def montar(df_parte, bloco_num):
        entradas = codificador.transform(df_parte)
        entradas['numericas'] = bloco_num
        return entradas

    return (montar(treino, num_treino), montar(val, num_val), montar(teste, num_teste),
            codificador.tamanhos_, (pipeline_num, codificador))


def construir_modelo_embeddings(tamanhos_categoricas, n_numericas, n_classes,
                                n_camadas=2, unidades_iniciais=128, taxa_dropout=0.3,
                                learning_rate=1e-3, otimizador_nome='adamw'):
    """MLP que aprende um vetor (embedding) para cada categoria antes das camadas densas"""
    entradas, ramos = [], []

    for nome, tamanho in tamanhos_categoricas.items():
        entrada = keras.Input(shape=(1,), dtype='int32', name=nome)
        # regra pratica: quanto mais categorias, maior o vetor, com teto de 16
        dim_embedding = int(min(16, max(2, round(1.6 * tamanho ** 0.56))))
        vetor = layers.Embedding(input_dim=tamanho, output_dim=dim_embedding,
                                 name=f'embedding_{nome}')(entrada)
        ramos.append(layers.Flatten()(vetor))
        entradas.append(entrada)

    entrada_num = keras.Input(shape=(n_numericas,), name='numericas')
    entradas.append(entrada_num)
    ramos.append(entrada_num)

    # junta os embeddings com as numericas e segue com o mesmo funil do modelo base
    x = layers.Concatenate()(ramos)
    unidades = unidades_iniciais
    for _ in range(n_camadas):
        x = layers.Dense(unidades, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(taxa_dropout)(x)
        unidades = max(unidades // 2, n_classes)

    saida = layers.Dense(n_classes, activation='softmax')(x)
    modelo = keras.Model(inputs=entradas, outputs=saida)
    modelo.compile(optimizer=_montar_otimizador(otimizador_nome, learning_rate),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelo

### Métricas, callbacks e busca de hiperparâmetros

O `EarlyStopping` agora monitora a **própria métrica que o Optuna otimiza** (recall), e não a perda
de validação. Antes o modelo restaurado no fim do treino era o de menor perda, que não é
necessariamente o de maior recall — o que deixava o treino desalinhado do objetivo.

In [ ]:
def recall_em_k(y_true, probas, k):
    """acerta se a classe certa estiver entre as k mais provaveis"""
    # pega as k classes mais provaveis e checa se a certa esta entre elas
    topk = np.argsort(-probas, axis=1)[:, :k]
    acertos = np.any(topk == np.asarray(y_true).reshape(-1, 1), axis=1)
    return acertos.mean()


def pontuacao_recall_macro(y_true, probas):
    return recall_score(y_true, probas.argmax(axis=1), average='macro', zero_division=0)


def pontuacao_recall_macro_e_top2(y_true, probas):
    recall_macro = recall_score(y_true, probas.argmax(axis=1), average='macro', zero_division=0)
    return 0.5 * recall_macro + 0.5 * recall_em_k(y_true, probas, k=2)


class MetricaDeValidacao(keras.callbacks.Callback):
    """Calcula a metrica de interesse na validacao a cada epoca e grava em `logs`,
    pra que o EarlyStopping possa monitorar ela em vez da perda. Se receber um trial
    do Optuna, tambem reporta o valor e deixa o trial ser podado no meio do treino.

    O predict usa batch grande porque o padrao do Keras (32) deixa a validacao lenta:
    e esse predict por epoca que dominava o tempo de cada trial.
    """

    NOME_METRICA = 'val_pontuacao'

    def __init__(self, X_val, y_val, funcao_pontuacao, trial=None, batch_size=4096):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.funcao_pontuacao = funcao_pontuacao
        self.trial = trial
        self.batch_size = batch_size
        self.historico = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        probas = self.model.predict(self.X_val, verbose=0, batch_size=self.batch_size)
        pontuacao = self.funcao_pontuacao(self.y_val, probas)
        logs[self.NOME_METRICA] = pontuacao
        self.historico.append(float(pontuacao))
        if self.trial is not None:
            self.trial.report(pontuacao, epoch)
            if self.trial.should_prune():
                raise optuna.TrialPruned()


def montar_callbacks(X_val, y_val, funcao_pontuacao, paciencia, trial=None):
    """o callback da metrica vem primeiro: ele grava `val_pontuacao` no log que o
    EarlyStopping le logo em seguida"""
    metrica = MetricaDeValidacao(X_val, y_val, funcao_pontuacao, trial=trial)
    parada = keras.callbacks.EarlyStopping(
        monitor=MetricaDeValidacao.NOME_METRICA, mode='max',
        patience=paciencia, restore_best_weights=True,
    )
    return [metrica, parada], metrica


def criar_objetivo(X_treino, y_treino, X_val, y_val, n_classes, pesos_classe, funcao_pontuacao,
                   opcoes_batch_size, epocas_max, paciencia):
    def objetivo(trial):
        # espaco de busca que o Optuna vai explorar
        n_camadas = trial.suggest_int('n_camadas', 1, 3)
        unidades_iniciais = trial.suggest_categorical('unidades_iniciais', [32, 64, 128])
        usar_batchnorm = trial.suggest_categorical('usar_batchnorm', [True, False])
        taxa_dropout = trial.suggest_float('taxa_dropout', 0.0, 0.5)
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        otimizador_nome = trial.suggest_categorical('otimizador_nome', ['adam', 'adamw'])
        batch_size = trial.suggest_categorical('batch_size', opcoes_batch_size)

        keras.backend.clear_session()
        modelo = construir_modelo_base(
            dim_entrada=X_treino.shape[1], n_classes=n_classes, n_camadas=n_camadas,
            unidades_iniciais=unidades_iniciais, usar_batchnorm=usar_batchnorm,
            taxa_dropout=taxa_dropout, learning_rate=learning_rate, otimizador_nome=otimizador_nome,
        )

        callbacks, metrica = montar_callbacks(X_val, y_val, funcao_pontuacao, paciencia, trial=trial)
        modelo.fit(
            X_treino, y_treino,
            validation_data=(X_val, y_val),
            epochs=epocas_max, batch_size=batch_size, class_weight=pesos_classe,
            callbacks=callbacks, verbose=0,
        )
        # melhor valor visto na validacao durante o treino desse trial
        return max(metrica.historico) if metrica.historico else 0.0
    return objetivo

### Utilidades: pesos de classe, limiar, gif de treinamento e salvamento

In [ ]:
def calcular_pesos_classe(y_treino, suavizar=False):
    """peso inversamente proporcional a frequencia da classe.
    Com `suavizar=True` usa a raiz do inverso, que e menos agressivo - necessario na
    Rede 2, onde o inverso puro passaria de 100x de diferenca entre a maior e a menor classe."""
    contagens = pd.Series(y_treino).value_counts().sort_index()
    pesos = 1.0 / contagens.values
    if suavizar:
        pesos = np.sqrt(pesos)
    pesos = pesos / pesos.sum() * len(contagens)
    return {int(i): float(p) for i, p in zip(contagens.index, pesos)}, contagens


def escolher_limiar(y_val, probas_val):
    """Procura na VALIDACAO o limiar de probabilidade que maximiza o recall macro.
    O teste so e usado depois, uma vez, com o limiar ja definido."""
    limiares = np.linspace(0.05, 0.95, 91)
    pontuacoes = [
        recall_score(y_val, (probas_val[:, 1] >= t).astype(int), average='macro', zero_division=0)
        for t in limiares
    ]
    melhor = int(np.argmax(pontuacoes))
    return float(limiares[melhor]), float(pontuacoes[melhor]), limiares, np.array(pontuacoes)


def gif_de_treinamento(historico_keras, pontuacoes_val, caminho_gif, titulo, max_frames=60):
    """Gera um gif mostrando o treino epoca a epoca: a perda de um lado e a metrica
    monitorada do outro."""
    perda_treino = historico_keras.history['loss']
    perda_val = historico_keras.history.get('val_loss', [])
    n_epocas = len(perda_treino)
    passo = max(1, n_epocas // max_frames)
    quadros = list(range(1, n_epocas + 1, passo))
    if quadros[-1] != n_epocas:
        quadros.append(n_epocas)

    fig, (ax_perda, ax_metrica) = plt.subplots(1, 2, figsize=(11, 4))

    def desenhar(indice):
        k = quadros[indice]
        ax_perda.clear(); ax_metrica.clear()

        ax_perda.plot(range(1, k + 1), perda_treino[:k], label='treino')
        if perda_val:
            ax_perda.plot(range(1, k + 1), perda_val[:k], label='validacao')
        ax_perda.set_xlim(1, n_epocas)
        ax_perda.set_ylim(0, max(max(perda_treino), max(perda_val or [0])) * 1.1)
        ax_perda.set_xlabel('epoca'); ax_perda.set_ylabel('perda')
        ax_perda.set_title('Perda'); ax_perda.legend(loc='upper right')

        ax_metrica.plot(range(1, k + 1), pontuacoes_val[:k], color='tab:green')
        ax_metrica.set_xlim(1, n_epocas)
        ax_metrica.set_ylim(min(pontuacoes_val) * 0.95, max(pontuacoes_val) * 1.05)
        ax_metrica.set_xlabel('epoca'); ax_metrica.set_ylabel('pontuacao (validacao)')
        ax_metrica.set_title(f'Metrica monitorada - epoca {k}/{n_epocas}')

        fig.suptitle(titulo)
        fig.tight_layout()

    animacao = FuncAnimation(fig, desenhar, frames=len(quadros), interval=180)
    animacao.save(caminho_gif, writer=PillowWriter(fps=6))
    plt.close(fig)
    print(f"Gif salvo em: {caminho_gif}")
    return caminho_gif


def salvar_artefatos(objeto, caminho):
    """o joblib falha em alguns ambientes ao serializar o transformador customizado;
    nesse caso cai pro dill. Se nem o dill estiver disponivel, avisa e segue - o
    modelo Keras em si ja foi salvo separadamente, que e o essencial."""
    try:
        joblib.dump(objeto, caminho)
        return 'joblib'
    except Exception as erro:
        print(f"joblib nao deu conta ({erro}); tentando com dill.")
    try:
        import dill
        with open(caminho, 'wb') as arquivo:
            dill.dump(objeto, arquivo)
        return 'dill'
    except Exception as erro:
        print(f"AVISO: nao consegui salvar os artefatos de pre-processamento ({erro}). "
              f"O modelo Keras foi salvo normalmente.")
        return None


def carregar_artefatos(caminho):
    """carregamento simetrico ao salvamento acima"""
    try:
        return joblib.load(caminho)
    except Exception:
        import dill
        with open(caminho, 'rb') as arquivo:
            return dill.load(arquivo)

## 6. Divisão dos dados (uma só, para as duas redes)

Aqui tem uma mudança importante em relação a como o projeto estava antes. As duas redes usam agora
**a mesma divisão**: a Rede 2 herda os conjuntos da Rede 1, em vez de sortear os dela por conta.

O motivo é que antes uma empresa podia estar no *teste* da Rede 1 e no *treino* da Rede 2 ao mesmo
tempo. Enquanto as redes são avaliadas em separado isso não atrapalha, mas na hora de avaliar a
cascata inteira vira vazamento: o número da Rede 2 sairia inflado, porque parte das empresas
encaminhadas pela Rede 1 já teria sido vista no treino da Rede 2.

A divisão é estratificada pelo grupo temático, que já embute a target das duas redes: o rótulo
"Não contratou consultoria" é a classe negativa da Rede 1, e os outros 8 são as classes da Rede 2.
Com isso, as duas redes ficam com as proporções preservadas de uma vez só.

In [ ]:
# estratificar pelo grupo tematico resolve as duas redes ao mesmo tempo
treino_val, teste = train_test_split(
    df_limpo, test_size=0.15, stratify=df_limpo['target_grupo'], random_state=SEED
)
treino, validacao = train_test_split(
    treino_val, test_size=0.15 / 0.85, stratify=treino_val['target_grupo'], random_state=SEED
)

print(f"Rede 1 (base completa)  -> treino: {len(treino):,} | validacao: {len(validacao):,} | teste: {len(teste):,}")

# a Rede 2 e o recorte de quem contratou, dentro de cada conjunto
treino_r2 = treino[treino['Contratou consultoria'] == 1].copy()
validacao_r2 = validacao[validacao['Contratou consultoria'] == 1].copy()
teste_r2 = teste[teste['Contratou consultoria'] == 1].copy()

print(f"Rede 2 (so contratantes) -> treino: {len(treino_r2):,} | validacao: {len(validacao_r2):,} | teste: {len(teste_r2):,}")

# confere que nenhuma empresa aparece em mais de um conjunto
assert set(treino.index).isdisjoint(teste.index) and set(validacao.index).isdisjoint(teste.index)
print("\nSem sobreposicao entre treino, validacao e teste.")

## 7. Rede 1 — Propensão a contratar consultoria

Problema binário sobre as 207.611 empresas. Alvo: `Contratou consultoria`. Sete variáveis: três
numéricas (atendimentos PJ, projetos atendidos, idade da empresa) e quatro categóricas (segmento,
município, tipo de estabelecimento, porte).

In [ ]:
COLUNAS_MODELO = FEATURES_NUMERICAS + FEATURES_CATEGORICAS

y_treino_r1 = treino['Contratou consultoria'].values
y_val_r1 = validacao['Contratou consultoria'].values
y_teste_r1 = teste['Contratou consultoria'].values

pre_processador_r1 = montar_pre_processador(FEATURES_NUMERICAS, FEATURES_CATEGORICAS)

# fit so no treino; validacao e teste apenas transformam (evita vazamento)
X_treino_r1 = pre_processador_r1.fit_transform(treino[COLUNAS_MODELO]).values.astype('float32')
X_val_r1 = pre_processador_r1.transform(validacao[COLUNAS_MODELO]).values.astype('float32')
X_teste_r1 = pre_processador_r1.transform(teste[COLUNAS_MODELO]).values.astype('float32')

print(f"Shapes: treino {X_treino_r1.shape} | validacao {X_val_r1.shape} | teste {X_teste_r1.shape}")

Num problema binário, chutar sempre a classe majoritária já dá 0,50 de recall macro. Então a Rede 1
precisa passar bem de 0,50 pra mostrar que aprendeu alguma coisa.

In [ ]:
# baseline ingenuo, so pra ter uma referencia de comparacao
for estrategia in ['most_frequent', 'stratified']:
    dummy = DummyClassifier(strategy=estrategia, random_state=SEED).fit(X_treino_r1, y_treino_r1)
    recall_dummy = recall_score(y_teste_r1, dummy.predict(X_teste_r1), average='macro', zero_division=0)
    print(f"Baseline ({estrategia}): recall macro (teste) = {recall_dummy:.4f}")

In [ ]:
pesos_classe_r1, contagens_r1 = calcular_pesos_classe(y_treino_r1)

for indice, nome in [(0, 'Nao contratou'), (1, 'Contratou')]:
    print(f"{nome:15s} | n treino = {contagens_r1.get(indice, 0):7,} | peso = {pesos_classe_r1[indice]:.4f}")

### Busca de hiperparâmetros (Rede 1)

O Optuna testa combinações de arquitetura e treino, guiado pelos resultados anteriores. O
`MedianPruner` interrompe no meio do treino os trials que já estão piores que a mediana, o que
libera tempo pra testar mais configurações.

In [ ]:
N_TRIALS_R1 = 20

objetivo_r1 = criar_objetivo(
    X_treino_r1, y_treino_r1, X_val_r1, y_val_r1,
    n_classes=2, pesos_classe=pesos_classe_r1, funcao_pontuacao=pontuacao_recall_macro,
    opcoes_batch_size=[128, 256, 512], epocas_max=60, paciencia=8,
)

estudo_r1 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    # o MedianPruner corta no meio do treino os trials que ja estao piores que a mediana
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)
estudo_r1.optimize(objetivo_r1, n_trials=N_TRIALS_R1, show_progress_bar=True)

melhores_params_r1 = estudo_r1.best_params
print("\nMelhores hiperparametros (Rede 1):")
for chave, valor in melhores_params_r1.items():
    print(f"  {chave:20s} = {valor}")
print(f"\nMelhor recall macro na validacao: {estudo_r1.best_value:.4f}")
print(f"Trials concluidos: {len([t for t in estudo_r1.trials if t.state.name == 'COMPLETE'])} | "
      f"podados: {len([t for t in estudo_r1.trials if t.state.name == 'PRUNED'])}")

### Modelo base — treinamento final

In [ ]:
keras.backend.clear_session()
modelo_r1_base = construir_modelo_base(
    dim_entrada=X_treino_r1.shape[1], n_classes=2,
    n_camadas=melhores_params_r1['n_camadas'],
    unidades_iniciais=melhores_params_r1['unidades_iniciais'],
    usar_batchnorm=melhores_params_r1['usar_batchnorm'],
    taxa_dropout=melhores_params_r1['taxa_dropout'],
    learning_rate=melhores_params_r1['learning_rate'],
    otimizador_nome=melhores_params_r1['otimizador_nome'],
)
modelo_r1_base.summary()

# retreino final com a melhor configuracao: mais epocas e mais paciencia que na busca
callbacks_r1, metrica_r1 = montar_callbacks(X_val_r1, y_val_r1, pontuacao_recall_macro, paciencia=15)
historico_r1_base = modelo_r1_base.fit(
    X_treino_r1, y_treino_r1,
    validation_data=(X_val_r1, y_val_r1),
    epochs=100, batch_size=melhores_params_r1['batch_size'],
    class_weight=pesos_classe_r1, callbacks=callbacks_r1, verbose=1,
)

In [ ]:
from IPython.display import Image, display

# gif do treino: perda de um lado, recall macro de validacao do outro
caminho_gif_r1 = gif_de_treinamento(
    historico_r1_base, metrica_r1.historico,
    os.path.join(PASTA_FIGURAS, 'treino_rede1_base.gif'),
    'Rede 1 (modelo base) - evolucao do treinamento',
)
display(Image(filename=caminho_gif_r1))

### Escolha do limiar de decisão

Com as probabilidades em mãos, ainda falta decidir a partir de qual valor a empresa é sinalizada
como propensa. O padrão (`argmax`, equivalente a 0,5) não é necessariamente o melhor para uma base
tão desbalanceada. O limiar é escolhido **na validação** e só depois aplicado ao teste, uma única
vez — assim o número final continua sendo uma estimativa honesta.

In [ ]:
probas_val_r1 = modelo_r1_base.predict(X_val_r1, verbose=0, batch_size=4096)
limiar_r1, recall_val_r1, limiares, curva_limiar = escolher_limiar(y_val_r1, probas_val_r1)

print(f"Limiar escolhido na validacao: {limiar_r1:.2f} (recall macro = {recall_val_r1:.4f})")
print(f"Para comparacao, o limiar padrao 0.50 daria recall macro = "
      f"{recall_score(y_val_r1, (probas_val_r1[:, 1] >= 0.5).astype(int), average='macro', zero_division=0):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(limiares, curva_limiar)
plt.axvline(limiar_r1, color='tab:red', linestyle='--', label=f'limiar escolhido = {limiar_r1:.2f}')
plt.xlabel('limiar de probabilidade'); plt.ylabel('recall macro (validacao)')
plt.title('Escolha do limiar de decisao - Rede 1')
plt.legend(); plt.tight_layout(); plt.show()

### Avaliação no teste

In [ ]:
def avaliar_rede1(modelo_ou_probas, X_teste, y_teste, limiar, nome):
    """aplica o limiar escolhido na validacao e devolve as metricas no teste"""
    probas = (modelo_ou_probas if isinstance(modelo_ou_probas, np.ndarray)
              else modelo_ou_probas.predict(X_teste, verbose=0, batch_size=4096))
    y_pred = (probas[:, 1] >= limiar).astype(int)

    resultado = {
        'modelo': nome,
        'limiar': round(limiar, 2),
        'recall macro': round(recall_score(y_teste, y_pred, average='macro', zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_teste, probas[:, 1]), 4),
        'recall (Contratou)': round(recall_score(y_teste, y_pred, pos_label=1, zero_division=0), 4),
    }
    verdadeiros_positivos = int(((y_pred == 1) & (y_teste == 1)).sum())
    sinalizados = int((y_pred == 1).sum())
    resultado['precisao (Contratou)'] = round(verdadeiros_positivos / max(sinalizados, 1), 4)
    resultado['empresas sinalizadas'] = sinalizados
    return resultado, probas, y_pred


resultado_r1_base, probas_teste_r1_base, y_pred_r1_base = avaliar_rede1(
    modelo_r1_base, X_teste_r1, y_teste_r1, limiar_r1, 'Base (frequencia)'
)

print(classification_report(y_teste_r1, y_pred_r1_base,
                            target_names=['Nao contratou', 'Contratou'], zero_division=0))
pd.DataFrame([resultado_r1_base]).set_index('modelo')

In [ ]:
# matriz de confusao e curva ROC lado a lado
fig, eixos = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay(confusion_matrix(y_teste_r1, y_pred_r1_base),
                       display_labels=['Nao contratou', 'Contratou']).plot(
    ax=eixos[0], colorbar=False, cmap='Blues')
eixos[0].set_title('Matriz de confusao - Rede 1 (modelo base)')

fpr, tpr, _ = roc_curve(y_teste_r1, probas_teste_r1_base[:, 1])
eixos[1].plot(fpr, tpr, label=f"Rede 1 (AUC = {resultado_r1_base['ROC-AUC']:.3f})")
eixos[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='chute aleatorio')
eixos[1].set_xlabel('taxa de falsos positivos'); eixos[1].set_ylabel('taxa de verdadeiros positivos')
eixos[1].set_title('Curva ROC - Rede 1'); eixos[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'rede1_matriz_roc.png'), dpi=120)
plt.show()

### Modelo com embeddings (Rede 1)

Mesma tarefa, arquitetura diferente: cada município, segmento, porte e tipo de estabelecimento vira
um vetor que a rede aprende junto com o resto dos pesos. Para a comparação ser justa, os dois
modelos veem exatamente as mesmas empresas nos mesmos conjuntos. Os hiperparâmetros aqui são fixos
(sem nova busca do Optuna), pra manter o tempo de execução sob controle.

In [ ]:
def montar_entradas_embeddings(df_parte, features_numericas, artefatos_emb):
    """monta o dicionario de entradas do modelo com embeddings pra um recorte qualquer da base"""
    pipeline_num, codificador = artefatos_emb
    entradas = codificador.transform(df_parte)
    entradas['numericas'] = pipeline_num.transform(df_parte[features_numericas]).astype('float32')
    return entradas


ent_treino_r1, ent_val_r1, ent_teste_r1, tamanhos_r1, artefatos_emb_r1 = preparar_dados_embeddings(
    treino, validacao, teste, FEATURES_NUMERICAS, FEATURES_CATEGORICAS
)

print("Vocabulario de cada variavel categorica (inclui o indice 0 de categoria desconhecida):")
for nome, tamanho in tamanhos_r1.items():
    print(f"  {nome:26s} {tamanho:>5} categorias")

In [ ]:
keras.backend.clear_session()
modelo_r1_emb = construir_modelo_embeddings(
    tamanhos_categoricas=tamanhos_r1, n_numericas=len(FEATURES_NUMERICAS), n_classes=2,
    n_camadas=2, unidades_iniciais=128, taxa_dropout=0.3,
    learning_rate=1e-3, otimizador_nome='adamw',
)
modelo_r1_emb.summary()

callbacks_emb_r1, metrica_emb_r1 = montar_callbacks(
    ent_val_r1, y_val_r1, pontuacao_recall_macro, paciencia=10
)
historico_r1_emb = modelo_r1_emb.fit(
    ent_treino_r1, y_treino_r1,
    validation_data=(ent_val_r1, y_val_r1),
    epochs=60, batch_size=512, class_weight=pesos_classe_r1,
    callbacks=callbacks_emb_r1, verbose=1,
)

In [ ]:
# o limiar deste modelo tambem e escolhido na validacao
probas_val_r1_emb = modelo_r1_emb.predict(ent_val_r1, verbose=0, batch_size=4096)
limiar_r1_emb, recall_val_r1_emb, _, _ = escolher_limiar(y_val_r1, probas_val_r1_emb)
print(f"Limiar escolhido na validacao: {limiar_r1_emb:.2f} (recall macro = {recall_val_r1_emb:.4f})")

probas_teste_r1_emb = modelo_r1_emb.predict(ent_teste_r1, verbose=0, batch_size=4096)
resultado_r1_emb, _, y_pred_r1_emb = avaliar_rede1(
    probas_teste_r1_emb, None, y_teste_r1, limiar_r1_emb, 'Embeddings'
)

comparacao_r1 = pd.DataFrame([resultado_r1_base, resultado_r1_emb]).set_index('modelo')
print("\nComparacao no conjunto de teste - Rede 1:")
comparacao_r1

In [ ]:
MELHOR_R1 = comparacao_r1['recall macro'].idxmax()
print(f"Modelo escolhido para a cascata: {MELHOR_R1}")

modelo_r1_final = modelo_r1_base if MELHOR_R1 == 'Base (frequencia)' else modelo_r1_emb
limiar_r1_final = limiar_r1 if MELHOR_R1 == 'Base (frequencia)' else limiar_r1_emb


def prever_r1(df_parte):
    """probabilidades da Rede 1 para um recorte qualquer da base, usando o modelo vencedor"""
    if MELHOR_R1 == 'Base (frequencia)':
        X = pre_processador_r1.transform(df_parte[COLUNAS_MODELO]).values.astype('float32')
        return modelo_r1_final.predict(X, verbose=0, batch_size=4096)
    entradas = montar_entradas_embeddings(df_parte, FEATURES_NUMERICAS, artefatos_emb_r1)
    return modelo_r1_final.predict(entradas, verbose=0, batch_size=4096)


# salva o modelo no formato nativo do Keras e o pre-processador junto,
# senao nao da pra aplicar o modelo em dados novos (o app web usa esses arquivos)
modelo_r1_final.save(os.path.join(PASTA_MODELOS, 'rede1_propensao.keras'))
formato = salvar_artefatos({
    'tipo_modelo': MELHOR_R1,
    'limiar': limiar_r1_final,
    'pre_processador': pre_processador_r1,
    'artefatos_embeddings': artefatos_emb_r1,
    'features_numericas': FEATURES_NUMERICAS,
    'features_categoricas': FEATURES_CATEGORICAS,
    'melhores_hiperparametros': melhores_params_r1,
}, os.path.join(PASTA_MODELOS, 'rede1_artefatos.joblib'))
print(f"Rede 1 salva em {PASTA_MODELOS}/ (artefatos serializados com {formato})")

## 8. Rede 2 — Qual consultoria recomendar

Agora só as empresas que contrataram, dentro dos mesmos conjuntos definidos na seção 6. São 8
classes (os grupos temáticos) e as mesmas 7 variáveis da Rede 1.

In [ ]:
# o codificador e ajustado com todos os grupos, pra garantir os 8 rotulos mesmo que
# algum deles fique raro em um dos conjuntos
le_target = LabelEncoder().fit(df_limpo.loc[df_limpo['Contratou consultoria'] == 1, 'target_grupo'])
n_classes_r2 = len(le_target.classes_)

y_treino_r2 = le_target.transform(treino_r2['target_grupo'])
y_val_r2 = le_target.transform(validacao_r2['target_grupo'])
y_teste_r2 = le_target.transform(teste_r2['target_grupo'])

pre_processador_r2 = montar_pre_processador(FEATURES_NUMERICAS, FEATURES_CATEGORICAS)
X_treino_r2 = pre_processador_r2.fit_transform(treino_r2[COLUNAS_MODELO]).values.astype('float32')
X_val_r2 = pre_processador_r2.transform(validacao_r2[COLUNAS_MODELO]).values.astype('float32')
X_teste_r2 = pre_processador_r2.transform(teste_r2[COLUNAS_MODELO]).values.astype('float32')

# quantos exemplos de cada classe caem em cada conjunto
distribuicao_r2 = pd.DataFrame({
    'treino': pd.Series(y_treino_r2).value_counts().sort_index(),
    'validacao': pd.Series(y_val_r2).value_counts().sort_index(),
    'teste': pd.Series(y_teste_r2).value_counts().sort_index(),
}).fillna(0).astype(int)
distribuicao_r2.index = le_target.classes_
distribuicao_r2.index.name = 'grupo tematico'
distribuicao_r2

In [ ]:
# com 8 classes, um chute daria por volta de 0,125 de recall macro
for estrategia in ['most_frequent', 'stratified']:
    dummy = DummyClassifier(strategy=estrategia, random_state=SEED).fit(X_treino_r2, y_treino_r2)
    recall_dummy = recall_score(y_teste_r2, dummy.predict(X_teste_r2), average='macro', zero_division=0)
    print(f"Baseline ({estrategia}): recall macro (teste) = {recall_dummy:.4f}")

Os pesos de classe aqui usam a raiz quadrada do inverso da frequência, não o inverso puro. Com o
inverso puro a diferença entre o maior e o menor peso passaria de 100x, o que desestabiliza o
treino. Com a raiz, essa diferença cai pra cerca de 5x: ainda favorece as classes pequenas, mas sem
exagero.

In [ ]:
pesos_classe_r2, contagens_r2 = calcular_pesos_classe(y_treino_r2, suavizar=True)

for indice, nome in enumerate(le_target.classes_):
    print(f"{nome:35s} | n treino = {contagens_r2.get(indice, 0):4d} | peso = {pesos_classe_r2[indice]:.3f}")

### Busca de hiperparâmetros (Rede 2)

Aqui a métrica otimizada é a média entre recall macro e Recall@2, não só o recall macro: na prática
o time comercial pode sugerir mais de uma consultoria pra mesma empresa, então acertar dentro das 2
primeiras já resolve o problema de negócio.

In [ ]:
N_TRIALS_R2 = 40

objetivo_r2 = criar_objetivo(
    X_treino_r2, y_treino_r2, X_val_r2, y_val_r2,
    n_classes=n_classes_r2, pesos_classe=pesos_classe_r2,
    funcao_pontuacao=pontuacao_recall_macro_e_top2,
    opcoes_batch_size=[32, 64, 128], epocas_max=100, paciencia=10,
)

estudo_r2 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15),
)
estudo_r2.optimize(objetivo_r2, n_trials=N_TRIALS_R2, show_progress_bar=True)

melhores_params_r2 = estudo_r2.best_params
print("\nMelhores hiperparametros (Rede 2):")
for chave, valor in melhores_params_r2.items():
    print(f"  {chave:20s} = {valor}")
print(f"\nMelhor pontuacao na validacao (0.5*recall_macro + 0.5*recall@2): {estudo_r2.best_value:.4f}")

### Modelo base — treinamento final

In [ ]:
keras.backend.clear_session()
modelo_r2_base = construir_modelo_base(
    dim_entrada=X_treino_r2.shape[1], n_classes=n_classes_r2,
    n_camadas=melhores_params_r2['n_camadas'],
    unidades_iniciais=melhores_params_r2['unidades_iniciais'],
    usar_batchnorm=melhores_params_r2['usar_batchnorm'],
    taxa_dropout=melhores_params_r2['taxa_dropout'],
    learning_rate=melhores_params_r2['learning_rate'],
    otimizador_nome=melhores_params_r2['otimizador_nome'],
)

callbacks_r2, metrica_r2 = montar_callbacks(
    X_val_r2, y_val_r2, pontuacao_recall_macro_e_top2, paciencia=20
)
historico_r2_base = modelo_r2_base.fit(
    X_treino_r2, y_treino_r2,
    validation_data=(X_val_r2, y_val_r2),
    epochs=150, batch_size=melhores_params_r2['batch_size'],
    class_weight=pesos_classe_r2, callbacks=callbacks_r2, verbose=1,
)

caminho_gif_r2 = gif_de_treinamento(
    historico_r2_base, metrica_r2.historico,
    os.path.join(PASTA_FIGURAS, 'treino_rede2_base.gif'),
    'Rede 2 (modelo base) - evolucao do treinamento',
)
display(Image(filename=caminho_gif_r2))

### Avaliação no teste

In [ ]:
def avaliar_rede2(probas, y_teste, nome):
    """recall macro mais as metricas top-k, que sao as que fazem sentido pro produto"""
    return {
        'modelo': nome,
        'recall macro': round(recall_score(y_teste, probas.argmax(axis=1), average='macro', zero_division=0), 4),
        'Recall@2': round(recall_em_k(y_teste, probas, k=2), 4),
        'Recall@3': round(recall_em_k(y_teste, probas, k=3), 4),
    }


probas_teste_r2_base = modelo_r2_base.predict(X_teste_r2, verbose=0, batch_size=4096)
resultado_r2_base = avaliar_rede2(probas_teste_r2_base, y_teste_r2, 'Base (frequencia)')

print(classification_report(y_teste_r2, probas_teste_r2_base.argmax(axis=1),
                            target_names=le_target.classes_, zero_division=0))
pd.DataFrame([resultado_r2_base]).set_index('modelo')

In [ ]:
# matriz 8x8: mostra com quais grupos o modelo confunde cada classe
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(confusion_matrix(y_teste_r2, probas_teste_r2_base.argmax(axis=1)),
                       display_labels=le_target.classes_).plot(
    ax=ax, xticks_rotation=90, colorbar=False, cmap='Blues')
plt.title('Matriz de confusao - Rede 2 (modelo base)')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'rede2_matriz.png'), dpi=120)
plt.show()

### Modelo com embeddings (Rede 2)

Mesma ideia da Rede 1: cada categoria vira um vetor aprendido. Aqui a base é pequena (poucos
milhares de empresas), então o modelo é mantido enxuto pra não sobreajustar.

In [ ]:
ent_treino_r2, ent_val_r2, ent_teste_r2, tamanhos_r2, artefatos_emb_r2 = preparar_dados_embeddings(
    treino_r2, validacao_r2, teste_r2, FEATURES_NUMERICAS, FEATURES_CATEGORICAS
)

keras.backend.clear_session()
modelo_r2_emb = construir_modelo_embeddings(
    tamanhos_categoricas=tamanhos_r2, n_numericas=len(FEATURES_NUMERICAS), n_classes=n_classes_r2,
    n_camadas=2, unidades_iniciais=64, taxa_dropout=0.4,
    learning_rate=1e-3, otimizador_nome='adamw',
)

callbacks_emb_r2, metrica_emb_r2 = montar_callbacks(
    ent_val_r2, y_val_r2, pontuacao_recall_macro_e_top2, paciencia=20
)
historico_r2_emb = modelo_r2_emb.fit(
    ent_treino_r2, y_treino_r2,
    validation_data=(ent_val_r2, y_val_r2),
    epochs=150, batch_size=64, class_weight=pesos_classe_r2,
    callbacks=callbacks_emb_r2, verbose=1,
)

probas_teste_r2_emb = modelo_r2_emb.predict(ent_teste_r2, verbose=0, batch_size=4096)
resultado_r2_emb = avaliar_rede2(probas_teste_r2_emb, y_teste_r2, 'Embeddings')

comparacao_r2 = pd.DataFrame([resultado_r2_base, resultado_r2_emb]).set_index('modelo')
print("\nComparacao no conjunto de teste - Rede 2:")
comparacao_r2

In [ ]:
MELHOR_R2 = comparacao_r2['Recall@2'].idxmax()
print(f"Modelo escolhido para a cascata: {MELHOR_R2}")

modelo_r2_final = modelo_r2_base if MELHOR_R2 == 'Base (frequencia)' else modelo_r2_emb


def prever_r2(df_parte):
    """probabilidades da Rede 2 para um recorte qualquer da base, usando o modelo vencedor"""
    if MELHOR_R2 == 'Base (frequencia)':
        X = pre_processador_r2.transform(df_parte[COLUNAS_MODELO]).values.astype('float32')
        return modelo_r2_final.predict(X, verbose=0, batch_size=4096)
    entradas = montar_entradas_embeddings(df_parte, FEATURES_NUMERICAS, artefatos_emb_r2)
    return modelo_r2_final.predict(entradas, verbose=0, batch_size=4096)


modelo_r2_final.save(os.path.join(PASTA_MODELOS, 'rede2_recomendacao.keras'))
formato = salvar_artefatos({
    'tipo_modelo': MELHOR_R2,
    'pre_processador': pre_processador_r2,
    'artefatos_embeddings': artefatos_emb_r2,
    'label_encoder_target': le_target,
    'features_numericas': FEATURES_NUMERICAS,
    'features_categoricas': FEATURES_CATEGORICAS,
    'melhores_hiperparametros': melhores_params_r2,
}, os.path.join(PASTA_MODELOS, 'rede2_artefatos.joblib'))
print(f"Rede 2 salva em {PASTA_MODELOS}/ (artefatos serializados com {formato})")

## 9. A cascata de ponta a ponta

Até aqui cada rede foi avaliada sozinha, e a Rede 2 só viu empresas que de fato contrataram. No
produto real não é assim: a Rede 2 só recebe quem a Rede 1 encaminhou. Se a Rede 1 deixa passar uma
empresa, a Rede 2 nunca chega a recomendar nada para ela — o erro se propaga.

A avaliação abaixo mede o sistema completo sobre o conjunto de teste, que nenhuma das duas redes
viu em treino. A pergunta é direta: **de todas as empresas do teste que realmente contrataram
alguma consultoria, para quantas o sistema inteiro acerta — ou seja, a Rede 1 sinaliza a empresa E a
Rede 2 coloca o tema certo entre as k sugestões?**

In [ ]:
# 1) Rede 1 decide quem e encaminhado
probas_cascata_r1 = prever_r1(teste)
sinalizadas = probas_cascata_r1[:, 1] >= limiar_r1_final

teste_avaliacao = teste.copy()
teste_avaliacao['sinalizada'] = sinalizadas

contratantes = teste_avaliacao[teste_avaliacao['Contratou consultoria'] == 1]
encaminhadas = contratantes[contratantes['sinalizada']]

total_contratantes = len(contratantes)
total_sinalizadas = int(sinalizadas.sum())

print(f"Empresas no teste:                       {len(teste_avaliacao):,}")
print(f"Sinalizadas pela Rede 1:                 {total_sinalizadas:,}")
print(f"Contratantes reais no teste:             {total_contratantes:,}")
print(f"Contratantes que a Rede 1 encaminhou:    {len(encaminhadas):,} "
      f"({len(encaminhadas)/total_contratantes*100:.1f}% - este e o teto da cascata)")
print(f"Contratantes perdidos pela Rede 1:       {total_contratantes - len(encaminhadas):,}")

In [ ]:
# 2) Rede 2 recomenda o tema para quem passou pela Rede 1
probas_cascata_r2 = prever_r2(encaminhadas)
y_encaminhadas = le_target.transform(encaminhadas['target_grupo'])

linhas = []
for k in [1, 2, 3]:
    acerto_isolado = recall_em_k(y_teste_r2, probas_teste_r2_base if MELHOR_R2 == 'Base (frequencia)'
                                 else probas_teste_r2_emb, k)
    acerto_encaminhadas = recall_em_k(y_encaminhadas, probas_cascata_r2, k)
    # cobertura ponta a ponta: precisa passar pela Rede 1 E acertar o tema na Rede 2
    cobertura = acerto_encaminhadas * len(encaminhadas) / total_contratantes
    linhas.append({
        'k': k,
        'Rede 2 isolada (Recall@k)': round(acerto_isolado, 4),
        'Rede 2 sobre os encaminhados': round(acerto_encaminhadas, 4),
        'cascata ponta a ponta': round(cobertura, 4),
    })

tabela_cascata = pd.DataFrame(linhas).set_index('k')
print("A ultima coluna e a metrica que descreve o produto de verdade.\n")
tabela_cascata

## 10. Discussão dos resultados

*(preencher com os números da execução: comparação das duas arquiteturas em cada rede, e o
resultado da cascata)*

## 11. Limitações

O grupo "Gestão Financeira e Estratégica" é o mais heterogêneo do agrupamento — junta financeiro,
planejamento, jurídico e gestão de pessoas. Essas consultorias provavelmente não compartilham o
mesmo padrão de contratação, e o agrupamento acabou misturando coisas diferentes.

As classes menores da Rede 2 têm poucas amostras (a menor tem 97 empresas no total, contra mais de
2.300 na maior), então o recall por classe nessas categorias oscila bastante entre execuções.

A precisão da Rede 1 na classe "Contratou" é baixa por escolha: priorizamos recall para não perder
oportunidades reais. Na prática isso significa que a maior parte das empresas sinalizadas não vai
contratar, e quem usar o modelo precisa saber disso.

O conjunto de variáveis é enxuto — sete no total, todas de perfil e de histórico de atendimento.
Não houve engenharia de atributos mais elaborada (cruzamentos entre segmento e município,
sazonalidade, histórico temporal por empresa). Boa parte do ganho que falta, principalmente na Rede
2, provavelmente está aí, e não em coletar dado novo.

A cascata propaga erro: a empresa que a Rede 1 não sinaliza nunca chega a receber uma recomendação.
A seção 9 mede exatamente o tamanho desse efeito.

A base é só do Sebrae/RN, então a generalização para outros estados ou períodos não foi testada.

## 12. Próximos passos

Engenharia de atributos sobre as variáveis existentes, revisão do agrupamento de "Gestão Financeira
e Estratégica", busca de hiperparâmetros também para o modelo com embeddings (aqui eles foram
fixados), e um teste do pipeline com dados de outros estados ou períodos.